In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"

# Narrative Framing Fine-tuning: Gemma 3-12B on Israel National News (Israeli Outlet)

## Overview

This notebook fine-tunes **Gemma 3-12B** on a corpus of **Israel National News** articles using
**QLoRA (4-bit quantized LoRA)**, and saves the resulting adapter **directly to Google Drive**.
## What We Accomplish

- **Domain Adaptation**: Continue-pretrain Gemma 3-12B on PNN's English-language news articles (title + body)
- **Memory-Efficient Training**: 4-bit quantization (QLoRA) + LoRA adapters, sized for a single Colab A100 (40GB)
- **Persistent Storage**: checkpoints and the final adapter save to Google Drive, in a separate
  folder from the other outlets' adapters so none overwrite each other

## Dataset

`israelnationalnews_filtered.csv` columns: `keyword`, `title`,
`label` (section), `link`, `date`, `article`.

## Requirements

- Colab **A100 (40GB, "High-RAM" GPU)**: go to Runtime → Change runtime type → A100
- A Hugging Face account with access to `google/gemma-3-12b-it` and a HF token set as the Colab
  secret `HF_TOKEN`
- A Google account with enough Drive space for the adapter


This cell installs/imports the libraries needed for QLoRA fine-tuning: `transformers`,
`peft` for LoRA, `bitsandbytes` for 4-bit quantization, `accelerate`, and `datasets`.
Random seeds are set for reproducibility, and warnings are suppressed for clean output.

In [ ]:
# Setup
# Uncomment on a fresh Colab runtime:
# !pip install -q -U transformers accelerate peft bitsandbytes datasets

from google.colab import drive, userdata
import os
import gc
from huggingface_hub import login
from datasets import Dataset
import pandas as pd
import random
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, PeftModel, get_peft_model, TaskType, prepare_model_for_kbit_training
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

This cell mounts Google Drive and defines `DRIVE_DIR`, the persistent location where
checkpoints and the final adapter will be saved throughout this notebook.

In [ ]:
# Mount Google Drive: all checkpoints and the final adapter save here
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/gemma3_12b_israelnews"
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"\u2705 Drive mounted. Saving everything under: {DRIVE_DIR}")

This cell loads the Israel National News article dataset (`israelnationalnews_filtered.csv`).
Rows missing a title or article body are dropped. The data is split into training (95%) and
evaluation (5%) sets using a fixed seed for reproducibility, and basic length statistics are
printed to sanity-check the data before training.

In [ ]:
# Data loading
CSV_PATH = "/content/drive/MyDrive/israelnationalnews_filtered.csv"


df = pd.read_csv(CSV_PATH)

df = df.dropna(subset=["title", "article"]).reset_index(drop=True)

print(f"\U0001F4F0 Loaded {len(df)} articles")
print(f"   Avg title length  : {df['title'].str.len().mean():.0f} chars")
print(f"   Avg article length: {df['article'].str.len().mean():.0f} chars")
print(f"   Max article length: {df['article'].str.len().max():.0f} chars")
print(f"\n   Sample title: {df['title'].iloc[0]}")

# Train/eval split
raw_dataset = Dataset.from_pandas(df).shuffle(seed=42)
split    = raw_dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

print(f"\n\U0001F4DA Training samples  : {len(train_ds)}")
print(f"\U0001F9EA Evaluation samples: {len(eval_ds)}")

This cell loads **Gemma 3-12B** in **4-bit (QLoRA)** quantization from Hugging Face.
4-bit loading via `bitsandbytes` shrinks the ~24GB bf16 footprint of a 12B model down to roughly
6-7GB, leaving plenty of headroom on a 40GB A100 for activations, LoRA adapters, and optimizer
state. `device_map="auto"` places the model on GPU automatically. You'll need to have accepted
Gemma 3's license on Hugging Face and be logged in (`huggingface-cli login` or the `HF_TOKEN`
Colab secret) since it's a gated model.

In [ ]:
# Hugging Face Login (required)
login(token=userdata.get('HF_TOKEN'))
print("\u2705 Logged in to Hugging Face")

In [ ]:
# Gemma 3-12B Model Setup (4-bit QLoRA)
# Model configuration
model_name = "google/gemma-3-12b-it"
device = torch.device("cuda")

print(f"\U0001F527 Loading model: {model_name}")
print(f"\U0001F4BB Device: {device}")

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Gemma 3 model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",)

print("\u2705 Gemma 3-12B model and tokenizer loaded successfully!")
print(f"\U0001F4CA Model parameters: {model.num_parameters():,}")
print("\U0001F525 Loaded in 4-bit (QLoRA) — ~6-7GB VRAM for weights instead of ~24GB in bf16")

This cell transforms the raw news articles into a plain next-token-prediction format
(`Title: ...\n\nArticle: ...`) and tokenizes them. `MAX_LENGTH` is passed through consistently
this time. Padding is deferred to the data collator so batches are only padded to
their own longest sequence, not the global max.

In [ ]:
# Article Dataset Presentation
MAX_LENGTH = 1024  # raise further if GPU memory allows; articles avg ~2,570 chars (~650 tokens)

def prepare_article_dataset(dataset, tokenizer, max_length=MAX_LENGTH):
    df = dataset.to_pandas() if hasattr(dataset, "to_pandas") else pd.DataFrame(dataset)

    texts = [
        f"Title: {row['title']}\n\nArticle: {row['article']}{tokenizer.eos_token}"
        for _, row in df.iterrows()]

    print(f"\U0001F504 Tokenizing {len(texts)} articles...")

    # NO padding=True, NO return_tensors="pt" (store as plain lists)
    tok = tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=max_length,)

    # Labels = input_ids
    return Dataset.from_dict({
        "input_ids"     : tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "labels"        : tok["input_ids"].copy(),})

print("\U0001F504 Preparing datasets...")
inst_train_ds = prepare_article_dataset(train_ds, tokenizer)
inst_eval_ds  = prepare_article_dataset(eval_ds,  tokenizer)

print(f"\n\u2705 Done")
print(f"   Training samples  : {len(inst_train_ds)}")
print(f"   Evaluation samples: {len(inst_eval_ds)}")

# Coverage check: what % of articles are being truncated?
token_lengths = [len(ids) for ids in inst_train_ds["input_ids"]]
truncated = sum(1 for l in token_lengths if l >= MAX_LENGTH)
print(f"\n\u26A0\uFE0F  Articles truncated at {MAX_LENGTH} tokens: "
      f"{truncated}/{len(token_lengths)} ({100*truncated/len(token_lengths):.1f}%)")
print(f"   Avg token length  : {sum(token_lengths)/len(token_lengths):.0f}")

This cell configures LoRA on top of the 4-bit quantized model (QLoRA). `prepare_model_for_kbit_training`
handles the housekeeping needed before attaching LoRA to a quantized model (casting norms to
fp32, enabling gradient checkpointing-compatible input grads, etc.). LoRA rank=16 / alpha=32 /
dropout=0.1 targets the standard attention + MLP projections. Training args use batch size 1 with
gradient accumulation (effective batch size 8) and bf16, which is what keeps a 12B model inside a
40GB A100.

In [ ]:
# LoRA Configuration and Training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Setup LoRA for efficient fine-tuning
lora_config = LoraConfig(
    # CAUSAL_LM: the model predicts the next token given previous tokens
    task_type=TaskType.CAUSAL_LM,

    # Set to False during training, True during inference
    inference_mode=False,

    # Rank of the low-rank decomposition matrices (A and B)
    r=16,

    # Scaling factor: LoRA_output = (alpha/r) * B * A * input
    lora_alpha=32,

    # Dropout for regularization
    lora_dropout=0.1,

    # Target the attention + MLP projections
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# Create LoRA model for instruction tuning
print("\U0001F527 Creating LoRA model...")
inst_model = get_peft_model(model, lora_config)
inst_model.enable_input_require_grads()   # required when gradient_checkpointing=True with PEFT
inst_model.print_trainable_parameters()

# Training arguments
inst_training_args = TrainingArguments(
    output_dir=f"{DRIVE_DIR}/instruction_results",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=1e-4,
    fp16=False,
    bf16=True,                          # native bf16 on A100
    optim="paged_adamw_8bit",
    logging_steps=10,
    logging_dir=None,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=[],
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,        # trades compute for memory (recomputes activations)
    max_grad_norm=1.0,
    dataloader_drop_last=True,)

# Data collator for language modeling with proper padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
    return_tensors="pt",)

print("\u2705 LoRA configuration and training setup completed")

This cell runs the actual fine-tuning via `Trainer`, evaluating every 50 steps.
The trained LoRA adapter and tokenizer are saved to disk when training completes.

In [ ]:
# Instruction Fine-Tuning
inst_trainer = Trainer(
    model=inst_model,
    args=inst_training_args,
    train_dataset=inst_train_ds,
    eval_dataset=inst_eval_ds,
    data_collator=data_collator,)

print("\U0001F3AF Starting fine-tuning of Gemma 3-12B...")

training_output = inst_trainer.train()

print("\u2705 Fine-tuning completed!")
print(f"\U0001F4CA Final training loss: {training_output.training_loss:.4f}")

print("\U0001F4BE Saving fine-tuned model to Drive...")
FINAL_SAVE_PATH = f"{DRIVE_DIR}/gemma3_12b_israelnews_finetuned"
inst_trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)
print(f"\u2705 Saved to {FINAL_SAVE_PATH}")

In [ ]:
checkpoints = sorted([
    d for d in os.listdir(f"{DRIVE_DIR}/instruction_results")
    if d.startswith("checkpoint-")
], key=lambda x: int(x.split("-")[1]))

print("Saved checkpoints:", checkpoints)
LAST_CHECKPOINT = f"{DRIVE_DIR}/instruction_results/{checkpoints[-1]}"
print(f"Using: {LAST_CHECKPOINT}")

# Load the LoRA adapter on top of the (already 4-bit) base model
inst_model = PeftModel.from_pretrained(model, LAST_CHECKPOINT)
inst_model.eval()

print(f"\u2705 Loaded fine-tuned model from {LAST_CHECKPOINT}")

# Save adapter
FINAL_ADAPTER_PATH = f"{DRIVE_DIR}/gemma3_12b_israelnews_final"
inst_model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)

print(f"\U0001F4BE Saved to {FINAL_ADAPTER_PATH}")
print("\U0001F449 Use this exact path as ADAPTER_PATH in the analysis notebook.")

This cell frees training-related memory (trainer, datasets, cached CUDA allocations) before
loading fresh copies of the models for evaluation.

In [ ]:
# Free training memory
del inst_trainer
del inst_train_ds, inst_eval_ds
del training_output
torch.cuda.empty_cache()
gc.collect()

print("\U0001F9F9 Training memory freed. Only base and fine-tuned models remain.")